# Movie Recommender System

In [29]:
import numpy as np
import pandas as pd
import ast
import hashlib
import pickle
import time
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')

In [30]:
# JSON parsing with caching
_eval_cache = {}

def cached_literal_eval(val):
    if pd.isna(val):
        return []
    key = hashlib.md5(str(val).encode()).hexdigest()
    if key in _eval_cache:
        return _eval_cache[key]
    try:
        result = ast.literal_eval(val)
        _eval_cache[key] = result
        return result
    except (ValueError, SyntaxError):
        return []

In [31]:
def load_and_parse_data(movies_path='tmdb_6000_movie_dataset.csv', credits_path='tmdb_6000_movie_credits.csv'):
    """Load data and parse JSON columns in parallel."""
    start_time = time.time()
    
    # Load data
    movies = pd.read_csv(movies_path, usecols=['tmdbId', 'title', 'overview', 'genres', 'keywords'])
    credits = pd.read_csv(credits_path, usecols=['tmdbId', 'cast', 'crew'])
    movies = movies.merge(credits, on='tmdbId')
    
    # Parse JSON columns in parallel
    for col in ['genres', 'keywords', 'cast', 'crew']:
        movies[col] = Parallel(n_jobs=-1)(delayed(cached_literal_eval)(val) for val in movies[col])
    
    # Extract director from crew
    movies['director'] = movies['crew'].apply(
        lambda x: [item['name'] for item in x if item.get('job') == 'Director'][:1] if x else []
    )
    
    # Keep genres for filtering
    movies['genres_list'] = movies['genres'].copy()
    
    print(f'Loaded and parsed data in {time.time()-start_time:.2f} sec')
    return movies

In [32]:
def combine_features_with_weights(row):
    """Combine features with implicit weighting."""
    features = []
    
    try:
        # Genres (highest weight - repeat 3 times)
        if isinstance(row['genres'], list):
            genres = []
            for g in row['genres']:
                if isinstance(g, dict):
                    genres.append(g.get('name', '').replace(' ', '').lower())
                else:
                    genres.append(str(g).replace(' ', '').lower())
            features.extend(genres * 3)
        
        # Director (high weight - repeat 2 times)
        if isinstance(row['director'], list):
            directors = [str(d).replace(' ', '').lower() for d in row['director']]
            features.extend(directors * 2)
        
        # Keywords (medium weight)
        if isinstance(row['keywords'], list):
            keywords = []
            for k in row['keywords']:
                if isinstance(k, dict):
                    keywords.append(k.get('name', '').replace(' ', '').lower())
                else:
                    keywords.append(str(k).replace(' ', '').lower())
            features.extend(keywords)
        
        # Cast (medium weight - top 3)
        if isinstance(row['cast'], list):
            cast = []
            for c in row['cast'][:3]:
                if isinstance(c, dict):
                    cast.append(c.get('name', '').replace(' ', '').lower())
                else:
                    cast.append(str(c).replace(' ', '').lower())
            features.extend(cast)
        
        # Overview (lower weight)
        if pd.notna(row['overview']) and isinstance(row['overview'], str):
            features.extend(row['overview'].lower().split())
            
    except Exception as e:
        print(f"Error processing row: {e}")
        return ""
    
    return ' '.join(features)


# Modified prepare_model function with better error handling
def prepare_model(movies, cache_file='recommender_cache.pkl'):
    """Prepare TF-IDF and similarity matrix with caching."""
    try:
        # Try to load from cache
        with open(cache_file, 'rb') as f:
            tfidf_matrix, similarity_matrix = pickle.load(f)
        print('Loaded cached TF-IDF and similarity matrix.')
        
    except FileNotFoundError:
        # Create new model
        print('Preparing TF-IDF and similarity matrix...')
        print("This may take 2-3 minutes...")
        
        # Combine features with weights
        print("Combining features...")
        movies['combined_features'] = movies.apply(combine_features_with_weights, axis=1)
        
        # Check if features were created
        print(f"Sample features: {movies['combined_features'].iloc[0][:200]}...")
        
        # Create TF-IDF vectors
        print("Creating TF-IDF vectors...")
        vectorizer = TfidfVectorizer(
            stop_words='english',
            max_features=5000,
            min_df=2
        )
        tfidf_matrix = vectorizer.fit_transform(movies['combined_features'])
        print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
        
        # Compute similarity
        print("Computing similarity matrix...")
        similarity_matrix = cosine_similarity(tfidf_matrix, dense_output=False)
        print(f"Similarity matrix shape: {similarity_matrix.shape}")
        
        # Cache the results
        with open(cache_file, 'wb') as f:
            pickle.dump((tfidf_matrix, similarity_matrix), f)
        print('✓ Prepared and cached TF-IDF & similarity matrix.')
        
    except Exception as e:
        print(f"Error in prepare_model: {e}")
        raise
    
    return tfidf_matrix, similarity_matrix


# Modified save function with explicit path and verification
def save_pickle_files(movies, similarity_matrix):
    """Save the required movie_list.pkl and similarity.pkl files."""
    print("\n" + "="*50)
    print("SAVING PICKLE FILES")
    print("="*50)
    
    try:
        # Create movie_list DataFrame
        movie_list = movies[['tmdbId', 'title', 'genres']].copy()
        
        # Convert genres to strings for the list
        movie_list['genres_display'] = movie_list['genres'].apply(
            lambda x: ', '.join([g.get('name', '') if isinstance(g, dict) else str(g) for g in x[:3]]) if isinstance(x, list) else ''
        )
        
        # Keep only needed columns
        movie_list = movie_list[['tmdbId', 'title', 'genres_display']]
        movie_list.columns = ['tmdbId', 'title', 'genres']
        
        # Save movie_list.pkl
        print("\nSaving movie_list.pkl...")
        with open('movie_list.pkl', 'wb') as f:
            pickle.dump(movie_list, f)
        print(f"✓ movie_list.pkl saved ({len(movie_list)} movies)")
        
        # Save similarity.pkl
        print("Saving similarity.pkl...")
        with open('similarity.pkl', 'wb') as f:
            pickle.dump(similarity_matrix, f)
        print(f"✓ similarity.pkl saved ({similarity_matrix.shape[0]}x{similarity_matrix.shape[1]})")
        
        # Verify files exist
        import os
        if os.path.exists('movie_list.pkl') and os.path.exists('similarity.pkl'):
            print("\n✓ Both files saved successfully!")
            print(f"Location: {os.getcwd()}")
        else:
            print("\n❌ Files not found after saving!")
            
    except Exception as e:
        print(f"\n❌ Error saving files: {e}")
        raise


# === MAIN EXECUTION ===
print("="*60)
print("MOVIE RECOMMENDER SYSTEM - PICKLE GENERATOR")
print("="*60)

# Load and parse data
print("\n1. Loading data...")
movies_df = load_and_parse_data()
print(f"   Loaded {len(movies_df)} movies")

# Prepare model (cached)
print("\n2. Building model...")
tfidf, similarity = prepare_model(movies_df)

# Save required pickle files
print("\n3. Saving pickle files...")
save_pickle_files(movies_df, similarity)

print("\n" + "="*60)
print("✅ PROCESS COMPLETE! Pickle files are ready.")
print("="*60)

# Test if files can be loaded
print("\nTesting file loading:")
try:
    test_movies = pickle.load(open('movie_list.pkl', 'rb'))
    test_sim = pickle.load(open('similarity.pkl', 'rb'))
    print("✓ Both files load successfully!")
    print(f"  - movie_list.pkl: {len(test_movies)} movies")
    print(f"  - similarity.pkl: {test_sim.shape}")
except Exception as e:
    print(f"❌ Error loading files: {e}")

MOVIE RECOMMENDER SYSTEM - PICKLE GENERATOR

1. Loading data...


FileNotFoundError: [Errno 2] No such file or directory: 'tmdb_6000_movie_dataset.csv'

In [ ]:
def prepare_model(movies, cache_file='recommender_cache.pkl'):
    """Prepare TF-IDF and similarity matrix with caching."""
    try:
        # Try to load from cache
        with open(cache_file, 'rb') as f:
            tfidf_matrix, similarity_matrix = pickle.load(f)
        print('Loaded cached TF-IDF and similarity matrix.')
        
    except FileNotFoundError:
        # Create new model
        print('Preparing TF-IDF and similarity matrix...')
        
        # Combine features with weights
        movies['combined_features'] = movies.apply(combine_features_with_weights, axis=1)
        
        # Create TF-IDF vectors
        vectorizer = TfidfVectorizer(
            stop_words='english',
            max_features=5000,  # Reduced for speed
            min_df=2  # Ignore very rare terms
        )
        tfidf_matrix = vectorizer.fit_transform(movies['combined_features'])
        
        # Compute similarity (sparse output for memory efficiency)
        similarity_matrix = cosine_similarity(tfidf_matrix, dense_output=False)
        
        # Cache the results
        with open(cache_file, 'wb') as f:
            pickle.dump((tfidf_matrix, similarity_matrix), f)
        print('Prepared and cached TF-IDF & similarity matrix.')
    
    return tfidf_matrix, similarity_matrix

In [ ]:
def recommend_movies_enhanced(title, movies, similarity_matrix, top_n=10, min_genre_overlap=1):
    """Recommend movies with genre filtering for accuracy."""
    if title not in movies['title'].values:
        return f"Movie '{title}' not found."
    
    idx = movies[movies['title'] == title].index[0]
    movie_genres = set(movies.iloc[idx]['genres_list'])
    
    # Get similarity scores
    sim_scores = list(enumerate(similarity_matrix[idx].toarray()[0]))
    
    # Filter by genre overlap
    filtered_scores = []
    for i, score in sim_scores:
        if i == idx:
            continue
        other_genres = set(movies.iloc[i]['genres_list'])
        genre_overlap = len(movie_genres.intersection(other_genres))
        
        if genre_overlap >= min_genre_overlap:
            filtered_scores.append((i, score, genre_overlap))
    
    # Sort by similarity score
    filtered_scores.sort(key=lambda x: x[1], reverse=True)
    
    # Get top recommendations
    recommendations = []
    for i, score, overlap in filtered_scores[:top_n]:
        recommendations.append({
            'title': movies.iloc[i]['title'],
            'similarity': round(score, 3),
            'genre_overlap': overlap,
            'genres': movies.iloc[i]['genres_list'][:3]  # Show top 3 genres
        })
    
    return recommendations

In [ ]:
def save_pickle_files(movies, similarity_matrix):
    """Save the required movie_list.pkl and similarity.pkl files."""
    print("\nSaving pickle files...")
    
    # Create movie_list DataFrame (format expected by many apps)
    movie_list = movies[['tmdbId', 'title', 'genres_list']].copy()
    movie_list['genres'] = movie_list['genres_list'].apply(lambda x: ', '.join(x[:3]))
    movie_list = movie_list[['tmdbId', 'title', 'genres']]
    
    # Save movie_list.pkl
    with open('movie_list.pkl', 'wb') as f:
        pickle.dump(movie_list, f)
    print("✓ movie_list.pkl saved")
    
    # Save similarity.pkl
    with open('similarity.pkl', 'wb') as f:
        pickle.dump(similarity_matrix, f)
    print("✓ similarity.pkl saved")
    
    # Also save a readable CSV for inspection
    movie_list.to_csv('movie_list.csv', index=False)
    print("✓ movie_list.csv saved (for reference)")

In [ ]:
def combine_features_with_weights(row):
    """Combine features with implicit weighting."""
    features = []
    
    # Genres (highest weight - repeat 3 times)
    # Extract 'name' from each genre dictionary
    if isinstance(row['genres'], list):
        genres = [g['name'].replace(' ', '').lower() if isinstance(g, dict) else str(g).replace(' ', '').lower() 
                  for g in row['genrics']]
        features.extend(genres * 3)
    
    # Director (high weight - repeat 2 times)
    if isinstance(row['director'], list):
        directors = [d.replace(' ', '').lower() if isinstance(d, str) else str(d).replace(' ', '').lower() 
                     for d in row['director']]
        features.extend(directors * 2)
    
    # Keywords (medium weight)
    if isinstance(row['keywords'], list):
        keywords = []
        for k in row['keywords']:
            if isinstance(k, dict):
                keywords.append(k['name'].replace(' ', '').lower())
            else:
                keywords.append(str(k).replace(' ', '').lower())
        features.extend(keywords)
    
    # Cast (medium weight - top 3)
    if isinstance(row['cast'], list):
        cast = []
        for c in row['cast'][:3]:
            if isinstance(c, dict):
                cast.append(c['name'].replace(' ', '').lower())
            else:
                cast.append(str(c).replace(' ', '').lower())
        features.extend(cast)
    
    # Overview (lower weight)
    if pd.notna(row['overview']) and isinstance(row['overview'], str):
        features.extend(row['overview'].lower().split())
    
    return ' '.join(features)

In [ ]:
# Quick recommendation function (for later use)
def quick_recommend(movie_title, top_n=5):
    """Quick recommendations using saved pickle files."""
    try:
        movie_list = pickle.load(open('movie_list.pkl', 'rb'))
        similarity = pickle.load(open('similarity.pkl', 'rb'))
        
        if movie_title not in movie_list['title'].values:
            return f"Movie '{movie_title}' not found."
        
        idx = movie_list[movie_list['title'] == movie_title].index[0]
        sim_scores = list(enumerate(similarity[idx].toarray()[0]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        recommendations = []
        for i, score in sim_scores[1:top_n+1]:
            recommendations.append({
                'title': movie_list.iloc[i]['title'],
                'similarity': round(score, 3),
                'genres': movie_list.iloc[i]['genres']
            })
        return recommendations
        
    except FileNotFoundError:
        return "Pickle files not found. Run the full pipeline first."

In [ ]:
# Example usage of quick_recommend after files are saved
print("\nTesting quick_recommend function:")
quick_recs = quick_recommend('Inception')
if isinstance(quick_recs, list):
    for rec in quick_recs:
        print(f"{rec['title']} (similarity: {rec['similarity']}) - {rec['genres']}")
else:
    print(quick_recs)


Testing quick_recommend function:
Tenet (similarity: 0.357) - Action, Thriller, Science Fiction
War of the Worlds : The Attack (similarity: 0.339) - Science Fiction, Mystery, Thriller
Congo (similarity: 0.293) - Action, Adventure, Drama
Awareness (similarity: 0.291) - Science Fiction, Mystery, Thriller
What Happened to Monday (similarity: 0.283) - Science Fiction, Thriller, Drama
